# GTA6 Clipper: Render (Colab, evening)

Downloads and edits every clip you approved (`yes` in the **Batch** tab's `approved` column)
and saves them to **Google Drive → GTA6 Clipper → <date> → <brand>**: an `.mp4` plus a `.txt`
with the title, description and hashtags to paste when posting.

**Every time:** `Runtime → Run all`, allow the two Google pop-ups (sheet + Drive).
About 1–2 min per clip. Faster with a GPU: `Runtime → Change runtime type → T4 GPU` (optional).

**One-time setup:** same `GH_TOKEN` secret as the Signals notebook (🔑 Secrets, Notebook access on).


In [ ]:
# 1. Get the latest code (fresh copy every run; the token is never printed or kept)
import base64, os, shutil, subprocess
from google.colab import userdata

REPO = "/content/gta6-clipper"
os.chdir("/content")  # never stand inside the folder we're about to delete
shutil.rmtree(REPO, ignore_errors=True)
token = userdata.get("GH_TOKEN").strip()
auth_header = "Authorization: Basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
p = subprocess.run(["git", "-c", f"http.extraHeader={auth_header}", "clone", "-q", "--depth", "1",
                    "https://github.com/VYOM222444/gta6-clipper.git", REPO],
                   capture_output=True, text=True)
del token, auth_header
if p.returncode != 0:
    msg = p.stderr.strip().splitlines()[-1] if p.stderr.strip() else "unknown error"
    raise SystemExit("Could not download the code: " + msg +
                     "\nCheck the GH_TOKEN secret (still valid? Contents: Read-only on gta6-clipper? Notebook access on?)")
print("Code ready:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"],
                                    capture_output=True, text=True).stdout)


In [ ]:
# 2. Install yt-dlp + Whisper (+ the GPU libraries, only when a GPU is attached)
!pip -q install -r /content/gta6-clipper/requirements-render.txt
import subprocess
if subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0:
    !pip -q install nvidia-cublas-cu12 nvidia-cudnn-cu12
else:
    print('No GPU: Whisper will run on the CPU (about 1-2 min per clip).')
import json
if json.load(open('/content/gta6-clipper/config/settings.json'))['render'].get('edits', {}).get('remove_music'):
    !pip -q install demucs==4.0.1


In [ ]:
# 3. Fonts for the captions and labels
import json, os, urllib.request
settings = json.load(open("/content/gta6-clipper/config/settings.json"))
fonts_dir = settings["render"]["fonts_dir"]
os.makedirs(fonts_dir, exist_ok=True)
for name, url in settings["render"]["fonts"].items():
    path = os.path.join(fonts_dir, name.replace(" ", "") + ".ttf")
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
print(os.listdir(fonts_dir))


In [ ]:
# 4. Sign in (sheet) and connect Google Drive (where the clips are saved)
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")
print("Ready.")


In [ ]:
# 5. Download + render every approved clip
import os, sys
os.chdir(REPO)
sys.path.insert(0, REPO)
os.environ["YTDLP_JS_RUNTIME"] = "node"
MAX_CLIPS = 3  # test the editing pack on 3 clips first; set 0 for the normal 10 per run
os.environ["CLIPPER_MAX_CLIPS"] = str(MAX_CLIPS)
from agents import render_batch
render_batch.main()
